In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pylab as pl
import pandas as pd
import scipy
import time
from matplotlib import rcParams
rcParams['figure.figsize'] = [15, 7]
import scipy.signal as signal

class Sensor:
    def __init__(self):
        self.tstep = 25e-9
        self.keV_per_area = None
        self.mV_per_ADC = 1000./4096.
        self.baseline = None
        self.bits = 12
        self.response = None
        self.bin_energies = None
        
class NaISensor(Sensor):
    def __init__(self, keV_per_area, baseline):
        super(Sensor, self).__init__()
        self.keV_per_area = keV_per_area
        self.baseline = baseline
        
        self.response = np.loadtxt('original/NaI_Response', usecols=1,dtype=float)
        self.response_bins = 1e3 * np.loadtxt('original/NaI_Response', usecols=0, dtype=float) # KeV
        
class PlasticSensor(Sensor):
    def __init__(self, keV_per_area, baseline):
        super(Sensor, self).__init__()
        self.keV_per_area = keV_per_area
        self.baseline = baseline
        self.response = np.loadtxt('original/LgPL_Response', usecols=1,dtype=float)
        self.response_bins = 1e3 * np.loadtxt('original/LgPL_Response', usecols=0, dtype=float) # KeV


class TGFTraceParam:
    def __init__(self, countrate, fwhm, mean, std, seconds_before_pulse, trace_length):
        self.countrate = countrate
        self.fwhm = fwhm
        self.TGF_duration = self.fwhm*3e-6
        self.mean = mean
        self.std = std
        self.seconds_before_pulse = seconds_before_pulse
        self.trace_length = trace_length
        

class OldProcessingParam:
    def __init__(self, thresh, int_i, dead_i, extend, escale):
        self.thresh = thresh
        self.int_i = int_i
        self.dead_i = dead_i     #deadtime = integration time
        self.extend = extend
        self.escale = escale
        
        
        

In [ ]:
#variables for creating the trace
countrate= 1e6
fwhm = 50.0
TGF_duration = fwhm*3e-6
counts = int(countrate*TGF_duration) #total counts incident on the detector
#fwhm = 50.0 #fwhm of the total TGF count distribution in units microseconds
#TGF_duration = fwhm*3e-6 #this is a really rough estimate to get a rough estimate of the count rate
#countrate = int(counts/TGF_duration)
mean = .7 #mean of the TGF trace distribution
std = .5 #detemines the amount of asymetry in the TGF trace distribution
seconds_before_pulse = 1e-9 # seconds before pulse
dt = 25e-9 #sampling rate in seconds. 40MHz
trace_length = 28000 #number of samples in a trace file (700us at 40MHz)
keV_per_area = .147 #determined by trial and error to match energy range of instrument
mV_per_ADC = 1000./4096.
specscale_keV = 5.0  #spectrum scaling i.e. keV/line in the spectrum file
baseline = 110
base_noise = 0 #units mV
bits = 12  #use 12 for doing listmode but use 10 to compare traces to real trace files

#variables for integrating trace pulses into listmode events
thresh = 8.0     #units of mV  this is the pulse trigger threshold
int_i = 50      #integ.ration time = 1.25 microsecs = 50 samples at 40MHz sampling
dead_i = int_i     #deadtime = integration time
extend = 1    #extendable dead time parameter
escale = .63  #being used to scale the pulse integration value to energy in keV. experimentally determined.


In [ ]:
# LIMITS
# signal level near noise floor. Really SNR must be > 100 for conventional deconvolution...
# Pulse magnitudes near saturation
# pileup causing saturation
# probability of discrete index coincidence...